In [25]:
import pandas as pd
import itertools
import networkx as nx
import matplotlib.pyplot as plt
import pickle
from collections import Counter
import numpy as np

In [27]:
authors_df = pd.read_csv('authors_23_03.csv') 

/var/folders/qq/hnwvbjqs21xg3srktb91gslr0000gn/T/ipykernel_12120/2501329752.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  authors_df = pd.read_csv('authors_23_03.csv')


In [29]:
authors_df.head()

,PMID,DescriptorName_UI,LastName,ForeName,AID,ORCID,Journal_JournalIssue_PubDate_MedlineDate,Journal_JournalIssue_PubDate_Year
0,439,D001943,Destailleur,G,2805531,NaN,NaN,1975.0
1,439,D001943,Vernaillen,P,1016682,NaN,NaN,1975.0
2,439,D001943,Pluygers,E,5665598,NaN,NaN,1975.0
3,630,D011471,Kolárová,M,7195573,NaN,NaN,1975.0
4,1889,D011471,de Voogt,H J,1222639,NaN,NaN,1975.0


# extracting Year and Decade

In [30]:
authors_df['Journal_JournalIssue_PubDate_Year']

0          1975.0
1          1975.0
2          1975.0
3          1975.0
4          1975.0
            ...  
2178186    2020.0
2178187    2020.0
2178188    2020.0
2178189    2020.0
2178190    2020.0
Name: Journal_JournalIssue_PubDate_Year, Length: 2178191, dtype: object

In [31]:
authors_df['Journal_JournalIssue_PubDate_Year'] = (
    authors_df['Journal_JournalIssue_PubDate_Year']
    .astype(str)  # Convert everything to string
    .str.extract('(\d{4})')  # Extract 4-digit years
)

In [32]:
## function for year extraction
def extract_year(row):
    if not pd.isnull(row['Journal_JournalIssue_PubDate_Year']):
        return int(row['Journal_JournalIssue_PubDate_Year'])
    elif not pd.isnull(row['Journal_JournalIssue_PubDate_MedlineDate']):
        # Extract year from MedlineDate (assuming it's formatted like "1980 Jan-Feb")
        medline_date = str(row['Journal_JournalIssue_PubDate_MedlineDate'])
        year = ''.join(filter(str.isdigit, medline_date[:4]))
        return int(year) if year.isdigit() else np.nan
    else:
        return np.nan

In [33]:
#apply
authors_df['Publication_Year'] = authors_df.apply(extract_year, axis=1)

#drop rows without a year
authors_df = authors_df.dropna(subset=['Publication_Year'])

authors_df['Publication_Year'] = authors_df['Publication_Year'].astype(int)

In [34]:
authors_df

,PMID,DescriptorName_UI,LastName,ForeName,AID,ORCID,Journal_JournalIssue_PubDate_MedlineDate,Journal_JournalIssue_PubDate_Year,Publication_Year
0,439,D001943,Destailleur,G,2805531,NaN,NaN,1975,1975
1,439,D001943,Vernaillen,P,1016682,NaN,NaN,1975,1975
2,439,D001943,Pluygers,E,5665598,NaN,NaN,1975,1975
3,630,D011471,Kolárová,M,7195573,NaN,NaN,1975,1975
4,1889,D011471,de Voogt,H J,1222639,NaN,NaN,1975,1975
...,...,...,...,...,...,...,...,...,...
2178186,32702310,D001943,Purushotham,Arnie,1316527,NaN,NaN,2020,2020
2178187,32702310,D001943,Nolte,Ellen,148517,NaN,NaN,2020,2020
2178188,32702310,D001943,Sullivan,Richard,4379387,NaN,NaN,2020,2020
2178189,32702310,D001943,Rachet,Bernard,7023995,NaN,NaN,2020,2020


In [35]:
def get_century_decade(year):
    if year <= 2010:
        start_decade = 2000
        end_decade = 2010
    elif year <= 2020:
        start_decade = 2011
        end_decade = 2020
    else:
        # Continue logic if more decades needed
        start_decade = ((year - 2011) // 10) * 10 + 2011
        end_decade = start_decade + 9
    return f"{start_decade}-{end_decade}"

In [36]:
authors_df['Century_Decade'] = authors_df['Publication_Year'].apply(get_century_decade)

In [37]:
authors_df['Century_Decade'].unique()

array(['2000-2010', '2011-2020', '2021-2030'], dtype=object)

In [38]:
authors_df.columns

Index(['PMID', 'DescriptorName_UI', 'LastName', 'ForeName', 'AID', 'ORCID',
       'Journal_JournalIssue_PubDate_MedlineDate',
       'Journal_JournalIssue_PubDate_Year', 'Publication_Year',
       'Century_Decade'],
      dtype='object')

# adding a disease mention column for easier lookup

In [39]:
authors_df['Mention'] = np.where(authors_df['DescriptorName_UI'].isin(['D013736', 'D011471']), 'prostate', 'breast')

# adding a column with the author full name

In [40]:
authors_df['author_name'] = authors_df['ForeName'] + ' ' + authors_df['LastName']

In [41]:
authors_df['author_name']

0              G Destailleur
1               P Vernaillen
2                 E Pluygers
3                 M Kolárová
4               H J de Voogt
                 ...        
2178186    Arnie Purushotham
2178187          Ellen Nolte
2178188     Richard Sullivan
2178189       Bernard Rachet
2178190        Ajay Aggarwal
Name: author_name, Length: 2178013, dtype: object

In [42]:
test = (authors_df.groupby("AID").size()[authors_df.groupby("AID").size() > 2].index)
filtered_authors_df = authors_df[authors_df["AID"].isin(test)]
print(filtered_authors_df)

             PMID DescriptorName_UI     LastName ForeName       AID  ORCID  \
2             439           D001943     Pluygers        E   5665598    NaN   
4            1889           D011471     de Voogt      H J   1222639    NaN   
5            2274           D001943        Sheth      N A   5949570    NaN   
6            2274           D001943     Ranadive      K J   7396392    NaN   
7            2274           D001943      Suraiya      J N    864796    NaN   
...           ...               ...          ...      ...       ...    ...   
2178185  32702310           D001943       Morris  Melanie   2873557    NaN   
2178186  32702310           D001943  Purushotham    Arnie   1316527    NaN   
2178188  32702310           D001943     Sullivan  Richard   4379387    NaN   
2178189  32702310           D001943       Rachet  Bernard   7023995    NaN   
2178190  32702310           D001943     Aggarwal     Ajay  16096345    NaN   

        Journal_JournalIssue_PubDate_MedlineDate  \
2          

In [43]:
print(authors_df['AID'].nunique())

711922


## removing trash data before 2002 and after 2020

In [ ]:
authors_df = authors_df[authors_df['Publication_Year']<2021] #for 2021 there was no complete data
authors_df = authors_df[authors_df['Publication_Year']>2001] #before the data is fucked

In [46]:
authors_df['Century_Decade'].unique()

array(['2000-2010', '2011-2020'], dtype=object)

## separate csvs for each decade

#group by decade
grouped_authors = authors_df.groupby('Century_Decade')['author_name'].apply(list).reset_index()

#create separate DataFrames for each century/decade and save them
for century_decade in grouped_authors['Century_Decade'].unique():
    authors_in_period = authors_df[authors_df['Century_Decade'] == century_decade]
    authors_in_period.to_csv(f"split_authors_{century_decade}.csv", index=False)

## save full data

In [47]:
authors_df.to_csv('authors_pre_cleaned.csv', index=False)